# MRSAC Farm Boundary Detection Pipeline

This notebook provides a unified pipeline for:
1.  **Setup**: Environment configuration and check.
2.  **Data Config**: Defining dataset paths dynamically.
3.  **Training**: Fine-tuning YOLOv8 on the farm dataset.
4.  **Evaluation**: Validating model performance.
5.  **Inference**: Running predictions and visualizing results.

## 1. Setup & Dependencies

In [ ]:
import os
import sys
from pathlib import Path
import cv2
import torch
from ultralytics import YOLO
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# Check for GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print(f"Torch version: {torch.__version__}")

## 2. Configuration
Define paths relative to the project root to ensure portability.

In [ ]:
# Define Project Root (Assuming notebook is in project_mrsac/notebooks/)
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent

print(f"Project Root: {PROJECT_ROOT}")

# Dataset Paths
DATASET_DIR = PROJECT_ROOT / "DATASET"
DATA_YAML = DATASET_DIR / "data.yaml"

# Model Paths
MODELS_DIR = PROJECT_ROOT / "models"
BASE_MODEL_PATH = MODELS_DIR / "yolov8n.pt"  # Start with Nano model
# SEG_MODEL_PATH = MODELS_DIR / "yolov8s-seg.pt" # Uncomment for segmentation

# Training Output
RUNS_DIR = PROJECT_ROOT / "runs"

# Check if critical files exist
if not DATA_YAML.exists():
    raise FileNotFoundError(f"Dataset config not found at: {DATA_YAML}")
if not BASE_MODEL_PATH.exists():
    print(f"Warning: Base model not found at {BASE_MODEL_PATH}. It will be downloaded automatically.")

print("✅ Configuration complete.")

## 3. Training
Train the YOLOv8 model on the dataset.

In [ ]:
# Load Model
model = YOLO(str(BASE_MODEL_PATH))  # Load pretrained model
# model = YOLO(str(SEG_MODEL_PATH)) # Use this for segmentation

# Train
# epochs: Number of training rounds
# imgsz: Image size
# batch: Batch size (reduce if running out of GPU memory)
# project: Where to save results

results = model.train(
    data=str(DATA_YAML),
    epochs=50,       # Adjust as needed
    imgsz=640,
    batch=8,
    device=device,
    project=str(RUNS_DIR / "detect"),
    name="train_pipeline"
)

print("✅ Training complete.")

## 4. Evaluation
Validate the best model on the test set.

In [ ]:
# Load the best trained model
best_model_path = RUNS_DIR / "detect" / "train_pipeline" / "weights" / "best.pt"

if best_model_path.exists():
    trained_model = YOLO(str(best_model_path))
    metrics = trained_model.val()  # Run validation
    print(f"mAP50: {metrics.box.map50}")
    print(f"mAP50-95: {metrics.box.map}")
else:
    print("❌ Best model weights not found. Did training finish successfully?")

## 5. Inference & Visualization
Run predictions on sample images from the test set.

In [ ]:
def visualize_prediction(image_path, model):
    # Run inference
    results = model(image_path)
    
    # Plot results
    for r in results:
        im_array = r.plot()  # plot a BGR numpy array of predictions
        im = Image.fromarray(im_array[..., ::-1])  # RGB PIL image
        plt.figure(figsize=(10, 10))
        plt.imshow(im)
        plt.axis('off')
        plt.show()

# Pick a random image from the test set
test_images = list((DATASET_DIR / "test" / "images").glob("*.jpg"))

if test_images:
    sample_image = test_images[0]
    print(f"Running inference on: {sample_image.name}")
    visualize_prediction(str(sample_image), trained_model)
else:
    print("No test images found.")